# Departed players and expanded FPL history

## Summary

The combined complete-fixture FPL panel contains **303,126 rows across 12 seasons**, including
**2,486 player codes absent from the captured current roster**. Two older seasons add 49,617 rows.
Partial older sources, annual totals, Understat appearances and team odds are separate tables.

This notebook checks dataset coverage and integrity. Saved cells were executed top to bottom in one fresh Python namespace because the sandbox blocks kernel sockets; they also run under a normal Python 3 kernel. A new prediction-model comparison has not
been run as part of this acquisition. The original seven gameweeks remain retrospective evidence.

## Context and methods

Inputs are committed CSVs, `coverage.json`, `validation.json`, and the raw package index beside this
notebook. Source inventories pin the primary repositories and Git blob identities. The collection
README describes joins, DST, missingness, provider differences and historical publication limits.

### Key assumptions

A full panel covers all 380 fixtures; it does not prove every historical deadline roster is complete.
Absence from today's FPL roster does not distinguish retirement, transfer and relegation. The old
R and JSON archives share an upstream FPL source. Tables at different grains overlap and cannot
be added to produce a headline sample size. Exact-value tables suit these coverage checks.

In [1]:
from pathlib import Path
import collections, csv, gzip, hashlib, json, re
here = Path.cwd()
ROOT = next(p for p in [here, *here.parents] if (p / 'history/extended/coverage.json').exists())
AREA = ROOT / 'history/extended'
coverage = json.loads((AREA / 'coverage.json').read_text())
validation = json.loads((AREA / 'validation.json').read_text())
def read_rows(name):
    with gzip.open(AREA / 'data' / name, 'rt', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))
print('Sources: history/extended/inventories and raw/index.json')
print('Validation checks recorded:', validation['checks_passed'])

Sources: history/extended/inventories and raw/index.json
Validation checks recorded: 7


## Data

### Complete FPL panel and departed-player coverage

In [2]:
panel = read_rows('fpl-complete-seasons.csv.gz')
keys = {(r['season'], r['player_id'], r['fixture_id']) for r in panel}
assert len(keys) == len(panel) == 303126
per_season = collections.Counter(r['season'] for r in panel)
print(f"{'Season':<12}{'Player/fixture rows':>22}")
for season, count in sorted(per_season.items()):
    print(f'{season:<12}{count:>22,}')
print('Total:', f'{len(panel):,}')
for key, value in coverage['complete_fpl_panel'].items():
    if key != 'base_panel_sha256': print(key + ':', value)

Season         Player/fixture rows
2014-15                     24,876
2015-16                     24,741
2016-17                     23,679
2017-18                     22,467
2018-19                     21,790
2019-20                     22,501
2020-21                     24,365
2021-22                     25,447
2022-23                     26,505
2023-24                     29,725
2024-25                     27,283
2025-26                     29,747
Total: 303,126
rows: 303126
seasons: 12
distinct_player_codes: 3020
absent_current_codes: 2486
absent_current_rows: 230128
original_rows: 253509
additional_complete_panel_rows: 49617
duplicate_player_fixture_keys: 0


The original ten seasons already included 2,090 absent player codes and 182,224 of their rows.
The larger panel adds 397 distinct codes overall and 396 absent codes. This is an expansion of an
existing all-player archive, rather than the removal of a current-player filter from old tests.

### Missingness and partial historical databases

In [3]:
partial = read_rows('partial-database-player-matches.csv.gz')
print(f"{'Season':<12}{'All rows':>12}{'Usable points/minutes':>25}{'Unknown points':>18}")
for season in sorted({r['season'] for r in partial}):
    group = [r for r in partial if r['season'] == season]
    usable = sum(r['complete_points_and_minutes'] == 'True' for r in group)
    missing = sum(r['total_points'] == '' for r in group)
    print(f'{season:<12}{len(group):>12,}{usable:>25,}{missing:>18,}')
    for r in group:
        if r['complete_points_and_minutes'] == 'True':
            assert r['minutes'] != '' and r['total_points'] != ''
            assert r['identity_conflict'] == 'False' and r['core_conflict'] == ''
print('Unknown targets remain blank; identity-conflicting rows are excluded from the usable count.')

Season          All rows    Usable points/minutes    Unknown points
2010-11           10,518                    9,790               400
2011-12           14,117                   10,897             2,761
2012-13           12,967                   11,366               866
2013-14           22,481                   13,296             9,039
2014-15           22,777                    1,891            20,876
2015-16           20,292                        0            20,292
Unknown targets remain blank; identity-conflicting rows are excluded from the usable count.


These are partial populations with overlapping source versions. The 2013/14 JSON extract has
17,805 rows over 291 fixtures and GW1–30. It overlaps the database observations. Later database
years also overlap the complete panels; the table above is not an additional independent sample.

## Results

### Annual totals and Understat coverage

In [4]:
annual = read_rows('annual-history.csv.gz')
understat = read_rows('understat-player-matches.csv.gz')
assert len({(r['season'], r['code']) for r in annual}) == len(annual)
assert all(re.fullmatch(r'20\d{2}-\d{2}', r['season']) for r in annual)
assert len({(r['understat_player_id'], r['id']) for r in understat}) == len(understat)
print('Unique FPL player-season totals:', f'{len(annual):,}')
print('Annual season labels:', min(r['season'] for r in annual), 'to', max(r['season'] for r in annual))
print('Unique Understat player matches:', f'{len(understat):,}')
print('Understat rows with an explicit stable FPL crosswalk:', sum(bool(r['fpl_code']) for r in understat))
print('Understat auxiliary annual rows:', coverage['understat']['auxiliary_player_seasons'])
print('Of those, partial 2025/26 rows:', coverage['understat']['auxiliary_partial_2025_26_rows'])
print('Team result/odds rows:', coverage['team_results']['rows'])

Unique FPL player-season totals: 9,994
Annual season labels: 2006-07 to 2025-26
Unique Understat player matches: 119,464
Understat rows with an explicit stable FPL crosswalk: 98312
Understat auxiliary annual rows: 37237
Of those, partial 2025/26 rows: 2422
Team result/odds rows: 12704


Annual FPL data covers 20 season labels but remains an incomplete older player population.
Understat match history spans multiple leagues and observed appearances, with incomplete player
coverage and date-only timing. Its xG and assists are different provider measurements. The
auxiliary annual 2025/26 file was captured in September 2025 and remains a partial-season file.

### Artifact hashes and reconciliation receipt

In [5]:
for artifact in coverage['artifacts']:
    data = (AREA / 'data' / artifact['file']).read_bytes()
    assert hashlib.sha256(data).hexdigest() == artifact['sha256']
print('Every derived artifact listed in coverage.json matches its recorded hash.')
for i, check in enumerate(validation['checks'], 1):
    print(str(i) + '.', check)
assert coverage['production_model_changed'] is False
assert coverage['live_feeds_changed'] is False

Every derived artifact listed in coverage.json matches its recorded hash.
1. All derived artifact hashes and row counts agree
2. 303126 unique player fixtures; 12 seasons; observed minutes/points; missing historical xG retained
3. All 24876 2014/15 R-data player/GW/minutes/points observations reconcile with the closed JSON source; shared FPL lineage
4. Annual identities and season aliases reconciled; no core numerical source conflicts
5. Unknown database targets and identity conflicts cannot be admitted as complete observed labels
6. Understat player/match keys unique; cross-provider IDs use explicit season-bound crosswalks
7. Every packaged raw object matches its captured source hash; archive paths are bounded


## Takeaways

The two additional full seasons and provider measurements give the next model comparison more
outcome history and potentially better lagged features. They do not establish a predictive gain.
Use chronological fitting, source-aware eligibility, compatible scoring eras and explicit missingness.
Use future unobserved weeks for prospective confirmation. Exact historical odds timing and
deadline roster/news/schedule vintages remain gaps. The production model and live feeds are unchanged.

Rerun integrity checks from the repository root with `python history/verify_extended.py`.
Reacquisition/conversion uses the isolated environment documented in the collection README.